# SHL Grammar Scoring: audio embeddings (Kaggle GPU)

Two pretrained speech models turn each clip into fixed-size vectors, averaged over time. They capture how the speaker sounds (pronunciation, rhythm, hesitation), which the transcript misses.

| Model | Output file | Shape |
|---|---|---|
| WavLM base plus | `outputs/emb_audio_wavlm.npz` | (clips, 13, 768): all layers |
| Whisper large-v3 encoder | `outputs/emb_audio_whisper.npz` | (clips, 9, 1280): layers 0, 4, ..., 32 |

Stored as float16 to keep the files small. Neither depends on the transcripts.

**How to run:** same as `kaggle_transcribe.ipynb` (competition data attached, GPU T4, Internet on, *Save & Run All*). About 25 minutes. Download both files from the *Output* tab into the repo's `outputs/` folder.

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import librosa
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoFeatureExtractor, AutoModel

WAVLM_MODEL = "microsoft/wavlm-base-plus"      # self-supervised speech model
WHISPER_ENCODER = "openai/whisper-large-v3"    # encoder of the Whisper speech recogniser
WHISPER_LAYERS = list(range(0, 33, 4))         # every 4th of the 33 hidden states, keeps the file small
SAMPLE_RATE = 16_000
WINDOW = 30 * SAMPLE_RATE                      # the Whisper encoder always sees 30 s of audio
SAMPLES_PER_FRAME = 320                        # one Whisper encoder frame = 20 ms

INPUT_DIR = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "No GPU found: set Accelerator to GPU in the notebook settings"
torch.manual_seed(42)

## Find the clips

Same lookup as `kaggle_transcribe.ipynb`, so rows come out in the same order (train then test, in CSV order).

In [ ]:
def list_files(root, suffixes):
    return sorted(Path(d) / f for d, _, names in os.walk(root, followlinks=True)
                  for f in names if Path(f).suffix.lower() in suffixes)

csv_paths = list_files(INPUT_DIR, {".csv"})
audio_by_stem = {}
for p in list_files(INPUT_DIR, {".wav"}):
    audio_by_stem.setdefault(p.stem, []).append(p)

def find_csv(name):
    matches = [p for p in csv_paths if p.name.lower() == name]
    assert len(matches) == 1, f"expected exactly one {name}, found {matches}"
    return matches[0]

def resolve_audio(value, split):
    candidates = audio_by_stem.get(Path(str(value)).stem, [])
    if len(candidates) > 1:
        candidates = [p for p in candidates
                      if any(split in part.lower() for part in p.relative_to(INPUT_DIR).parts[:-1])]
    assert len(candidates) == 1, f"cannot find a unique audio file for {value!r} ({split}): {candidates}"
    return candidates[0]

files = pd.concat([
    pd.DataFrame({"file_id": pd.read_csv(find_csv("train.csv"))["filename"].astype(str), "split": "train"}),
    pd.DataFrame({"file_id": pd.read_csv(find_csv("test.csv"))["filename"].astype(str), "split": "test"}),
], ignore_index=True)
files["path"] = [resolve_audio(f, s) for f, s in zip(files["file_id"], files["split"])]
print(files["split"].value_counts().to_string())

def load_audio(path):
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)  # 16 kHz mono, what both models expect
    return audio.astype(np.float32)

def save_npz(path, emb):
    # Row keys go with the matrix so the main notebook can verify the order.
    np.savez_compressed(path, file_id=files["file_id"].to_numpy(dtype=str),
                        split=files["split"].to_numpy(dtype=str), emb=emb)
    print(path.name, emb.shape, f"{path.stat().st_size / 1e6:.1f} MB")

## WavLM

WavLM turns every 20 ms of audio into a vector at each of its 13 layers. Each layer's vectors are averaged over the whole clip.

In [ ]:
extractor = AutoFeatureExtractor.from_pretrained(WAVLM_MODEL)
model = AutoModel.from_pretrained(WAVLM_MODEL).to(DEVICE).eval()
pooled_all = []
for path in tqdm(files["path"], desc="WavLM"):
    inputs = extractor(load_audio(path), sampling_rate=SAMPLE_RATE, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        hidden = model(**inputs, output_hidden_states=True).hidden_states
    pooled_all.append(torch.stack([h[0].mean(0) for h in hidden]).cpu().numpy().astype(np.float16))
save_npz(OUT_DIR / "emb_audio_wavlm.npz", np.stack(pooled_all))
del model
torch.cuda.empty_cache()

## Whisper encoder

Each clip is cut into 30 second windows (Whisper pads the last one with silence). For each kept layer, frame vectors are averaged over the real audio only, ignoring the padding, and windows are combined weighted by their length.

In [ ]:
extractor = AutoFeatureExtractor.from_pretrained(WHISPER_ENCODER)
encoder = AutoModel.from_pretrained(WHISPER_ENCODER).get_encoder().half().to(DEVICE).eval()
pooled_all = []
for path in tqdm(files["path"], desc="Whisper encoder"):
    audio = load_audio(path)
    windows = [audio[s:s + WINDOW] for s in range(0, len(audio), WINDOW)] or [audio]
    feats = extractor(windows, sampling_rate=SAMPLE_RATE, return_tensors="pt").input_features
    with torch.no_grad():
        hidden = encoder(feats.to(DEVICE, torch.float16), output_hidden_states=True).hidden_states
    n_frames = [max(1, int(np.ceil(len(w) / SAMPLES_PER_FRAME))) for w in windows]
    pooled = [sum(hidden[L][i, :n].float().sum(0) for i, n in enumerate(n_frames)) / sum(n_frames)
              for L in WHISPER_LAYERS]
    pooled_all.append(torch.stack(pooled).cpu().numpy().astype(np.float16))
save_npz(OUT_DIR / "emb_audio_whisper.npz", np.stack(pooled_all))